In [ ]:
import pandas as pd

In [ ]:
df_stats_A = pd.read_csv("df_form_A_with_emotion.csv")
df_stats_B = pd.read_csv("df_form_B_LLM_parse.csv")
df_stats_final = pd.read_csv("df_final_LLM_parse.csv")
df_model = pd.read_csv("df_model.csv").rename(columns={"record_index": "id"})

exclude_descriptions = [
    "No intention was found",
    "Intention was deleted",
    'Video has frozen, unable to progress without pressing "No intention" checkbox. Will send message with screenshot to prolific',
]

before_rows = len(df_stats_A) + len(df_stats_B) + len(df_stats_final) + len(df_model)
df_stats_A = df_stats_A[~df_stats_A['intention_description'].isin(exclude_descriptions)].copy()
df_stats_B = df_stats_B[~df_stats_B['intention_description'].isin(exclude_descriptions)].copy()
df_stats_final = df_stats_final[~df_stats_final['intention_description'].isin(exclude_descriptions)].copy()
df_model = df_model[~df_model['intention_description'].isin(exclude_descriptions)].copy()

after_rows = len(df_stats_A) + len(df_stats_B) + len(df_stats_final) + len(df_model)
removed_rows = before_rows - after_rows

print(f"Loaded df_form_A_LLM_parse.csv into df_stats_A: {len(df_stats_A)} rows x {df_stats_A.shape[1]} columns")
print(f"Loaded df_form_B_LLM_parse.csv into df_stats_B: {len(df_stats_B)} rows x {df_stats_B.shape[1]} columns")
print(f"Loaded df_final_LLM_parse.csv into df_stats_final: {len(df_stats_final)} rows x {df_stats_final.shape[1]} columns")
print(f"Loaded df_model.csv into df_model: {len(df_model)} rows x {df_model.shape[1]} columns")
print(f"Removed {removed_rows} rows total based on excluded intention_description values")

In [ ]:
import sys
import importlib

# Clear cached modules
if 'sentence_transformers' in sys.modules:
    del sys.modules['sentence_transformers']
if 'transformers' in sys.modules:
    del sys.modules['transformers']

from sentence_transformers import SentenceTransformer

In [ ]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# Load the all-MiniLM-L6-v2 model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Extract intention_description values and embed them
intention_descriptions = df_stats_A['intention_description'].dropna().tolist()
embeddings = model.encode(intention_descriptions)

print(f"Loaded model: all-MiniLM-L6-v2")
print(f"Number of descriptions embedded: {len(embeddings)}")
print(f"Embedding dimension: {embeddings.shape[1]}")

In [ ]:
import numpy as np

def overlap_similarity(x, y, eps=1e-8):
    dot = np.dot(x, y)
    denom = np.dot(x, x) + np.dot(y, y) - abs(dot) + eps
    return dot / denom

def hts_similarity(x, y, eps=1e-8):
    dot = np.dot(x, y)
    denom = np.dot(x, x) + np.dot(y, y) + eps
    return np.tanh(2 * dot / denom)

In [ ]:
import numpy as np
import plotly.express as px
import textwrap
import umap


def two_line_hover(text, max_chars=90):
    wrapped = textwrap.wrap(str(text), width=max_chars)
    if len(wrapped) <= 2:
        return '<br>'.join(wrapped)
    first = wrapped[0]
    second = ' '.join(wrapped[1:])
    return first + '<br>' + textwrap.fill(second, width=max_chars)

# Extract and embed descriptions from A, B, final, and model
print("Extracting and embedding descriptions from df_stats_A, df_stats_B, df_stats_final, and df_model...")
rows_A = df_stats_A[['id', 'intention_description']].dropna(subset=['id', 'intention_description'])
rows_B = df_stats_B[['id', 'intention_description']].dropna(subset=['id', 'intention_description'])
rows_final = df_stats_final[['id', 'intention_description']].dropna(subset=['id', 'intention_description'])
rows_model = df_model[['id', 'intention_description']].dropna(subset=['id', 'intention_description'])

descriptions_A = rows_A['intention_description'].tolist()
descriptions_B = rows_B['intention_description'].tolist()
descriptions_final = rows_final['intention_description'].tolist()
descriptions_model = rows_model['intention_description'].tolist()
ids_A = rows_A['id'].tolist()
ids_B = rows_B['id'].tolist()
ids_final = rows_final['id'].tolist()
ids_model = rows_model['id'].tolist()

embeddings_A = model.encode(descriptions_A)
embeddings_B = model.encode(descriptions_B)
embeddings_final = model.encode(descriptions_final)
embeddings_model = model.encode(descriptions_model)

# Combine all embeddings
all_embeddings = np.vstack([embeddings_A, embeddings_B, embeddings_final, embeddings_model]).astype(np.float32)
all_descriptions = descriptions_A + descriptions_B + descriptions_final + descriptions_model
all_ids = ids_A + ids_B + ids_final + ids_model
n_A = len(descriptions_A)
n_B = len(descriptions_B)
n_final = len(descriptions_final)
n_model = len(descriptions_model)
n_total = n_A + n_B + n_final + n_model

print(f"Total points: {n_total} (A: {n_A}, B: {n_B}, final: {n_final}, model: {n_model})")

# Build similarity matrix using a vectorized form of hts_similarity
print("Computing pairwise hts similarities...")
dot_matrix = all_embeddings @ all_embeddings.T
norms = np.sum(all_embeddings * all_embeddings, axis=1)
denom = norms[:, None] + norms[None, :] + 1e-8
sims = np.tanh(2.0 * dot_matrix / denom)
np.fill_diagonal(sims, 1.0)

# Convert similarity to distance for UMAP
dists = 1.0 - sims
dists = np.maximum(dists, 0.0)
np.fill_diagonal(dists, 0.0)

print("Running UMAP with precomputed distance matrix...")
reducer = umap.UMAP(
    metric='precomputed',
    random_state=42,
    n_neighbors=150,
    min_dist=0.25,
)
embedding_2d = reducer.fit_transform(dists)

# Prepare plot dataframe with source label
source = ['A'] * n_A + ['B'] * n_B + ['final'] * n_final + ['model'] * n_model
plot_df = pd.DataFrame({
    'x': embedding_2d[:, 0],
    'y': embedding_2d[:, 1],
    'id': all_ids,
    'description': [two_line_hover(value, max_chars=120) for value in all_descriptions],
    'source': source,
    'avg_similarity': sims.mean(axis=1)
})

fig = px.scatter(
    plot_df,
    x='x',
    y='y',
    color='source',
    category_orders={'source': ['final', 'A', 'B', 'model']},
    custom_data=['id', 'description', 'avg_similarity', 'source'],
    color_discrete_map={'A': '#609AC1', 'B': '#C8C85A', 'final': '#ADE09E', 'model': '#D07EBD'},
    title='UMAP: Intention Descriptions from A, B, final, and model',
    width=1000,
    height=800
)
fig.update_traces(
    marker=dict(size=8, opacity=0.7),
    hovertemplate='id=%{customdata[0]}<br>description=%{customdata[1]}<br>avg_similarity=%{customdata[2]:.3f}<br>source=%{customdata[3]}<extra></extra>'
)
fig.update_layout(
    hoverlabel=dict(align='left'),
    paper_bgcolor='white',
    plot_bgcolor='white',
    xaxis=dict(showgrid=False, zeroline=False),
    yaxis=dict(showgrid=False, zeroline=False),
)
fig.show()

print('UMAP projection complete.')

In [ ]:
import numpy as np
import plotly.express as px
import textwrap
import umap


def two_line_hover(text, max_chars=90):
    wrapped = textwrap.wrap(str(text), width=max_chars)
    if len(wrapped) <= 2:
        return '<br>'.join(wrapped)
    first = wrapped[0]
    second = ' '.join(wrapped[1:])
    return first + '<br>' + textwrap.fill(second, width=max_chars)

# Extract and embed intention_explanation from A, B, final, and model
print("Extracting and embedding intention_explanation from df_stats_A, df_stats_B, df_stats_final, and df_model...")
rows_A = df_stats_A[['id', 'intention_explanation']].dropna(subset=['id', 'intention_explanation'])
rows_B = df_stats_B[['id', 'intention_explanation']].dropna(subset=['id', 'intention_explanation'])
rows_final = df_stats_final[['id', 'intention_explanation']].dropna(subset=['id', 'intention_explanation'])
rows_model = df_model[['id', 'intention_explanation']].dropna(subset=['id', 'intention_explanation'])

explanations_A = rows_A['intention_explanation'].tolist()
explanations_B = rows_B['intention_explanation'].tolist()
explanations_final = rows_final['intention_explanation'].tolist()
explanations_model = rows_model['intention_explanation'].tolist()
ids_A = rows_A['id'].tolist()
ids_B = rows_B['id'].tolist()
ids_final = rows_final['id'].tolist()
ids_model = rows_model['id'].tolist()

embeddings_A = model.encode(explanations_A)
embeddings_B = model.encode(explanations_B)
embeddings_final = model.encode(explanations_final)
embeddings_model = model.encode(explanations_model)

# Combine all embeddings
all_embeddings = np.vstack([embeddings_A, embeddings_B, embeddings_final, embeddings_model]).astype(np.float32)
all_explanations = explanations_A + explanations_B + explanations_final + explanations_model
all_ids = ids_A + ids_B + ids_final + ids_model
n_A = len(explanations_A)
n_B = len(explanations_B)
n_final = len(explanations_final)
n_model = len(explanations_model)
n_total = n_A + n_B + n_final + n_model

print(f"Total points: {n_total} (A: {n_A}, B: {n_B}, final: {n_final}, model: {n_model})")

# Build similarity matrix using a vectorized form of hts_similarity
print("Computing pairwise hts similarities...")
dot_matrix = all_embeddings @ all_embeddings.T
norms = np.sum(all_embeddings * all_embeddings, axis=1)
denom = norms[:, None] + norms[None, :] + 1e-8
sims = np.tanh(2.0 * dot_matrix / denom)
np.fill_diagonal(sims, 1.0)

# Convert similarity to distance for UMAP
dists = 1.0 - sims
dists = np.maximum(dists, 0.0)
np.fill_diagonal(dists, 0.0)

print("Running UMAP with precomputed distance matrix...")
reducer = umap.UMAP(
    metric='precomputed',
    random_state=42,
    n_neighbors=150,
    min_dist=0.25,
)
embedding_2d = reducer.fit_transform(dists)

# Prepare plot dataframe with source label
source = ['A'] * n_A + ['B'] * n_B + ['final'] * n_final + ['model'] * n_model
plot_df = pd.DataFrame({
    'x': embedding_2d[:, 0],
    'y': embedding_2d[:, 1],
    'id': all_ids,
    'description': [two_line_hover(value, max_chars=120) for value in all_explanations],
    'source': source,
    'avg_similarity': sims.mean(axis=1)
})

fig = px.scatter(
    plot_df,
    x='x',
    y='y',
    color='source',
    category_orders={'source': ['final', 'A', 'B', 'model']},
    custom_data=['id', 'description', 'avg_similarity', 'source'],
    color_discrete_map={'A': '#609AC1', 'B': '#C8C85A', 'final': '#ADE09E', 'model': '#D07EBD'},
    title='UMAP: Intention Explanations from A, B, final, and model',
    width=1000,
    height=800
)
fig.update_traces(
    marker=dict(size=8, opacity=0.7),
    hovertemplate='id=%{customdata[0]}<br>description=%{customdata[1]}<br>avg_similarity=%{customdata[2]:.3f}<br>source=%{customdata[3]}<extra></extra>'
)
fig.update_layout(
    hoverlabel=dict(align='left'),
    paper_bgcolor='white',
    plot_bgcolor='white',
    xaxis=dict(showgrid=False, zeroline=False),
    yaxis=dict(showgrid=False, zeroline=False),
)
fig.show()

print('UMAP projection complete.')

In [ ]:
# import numpy as np
# import pandas as pd
# import plotly.express as px
# import textwrap
# import umap
# import ipywidgets as widgets
# from IPython.display import display, clear_output


# def two_line_hover(text, max_chars=120):
#     wrapped = textwrap.wrap(str(text), width=max_chars)
#     if len(wrapped) <= 2:
#         return '<br>'.join(wrapped)
#     first = wrapped[0]
#     second = ' '.join(wrapped[1:])
#     return first + '<br>' + textwrap.fill(second, width=max_chars)

# n_neighbors_values = [5, 10, 20, 50, 100, 200, 500]
# min_dist_values = [0, 0.1, 0.25, 0.5, 0.75, 0.99]

# projection_cache = {'description': {}, 'explanation': {}}


# def compute_projection_cache(rows_A, rows_B, rows_final, rows_model, text_column, cache_key, max_chars=120):
#     descriptions_A = rows_A[text_column].tolist()
#     descriptions_B = rows_B[text_column].tolist()
#     descriptions_final = rows_final[text_column].tolist()
#     descriptions_model = rows_model[text_column].tolist()
#     ids_A = rows_A['id'].tolist()
#     ids_B = rows_B['id'].tolist()
#     ids_final = rows_final['id'].tolist()
#     ids_model = rows_model['id'].tolist()

#     embeddings_A = model.encode(descriptions_A)
#     embeddings_B = model.encode(descriptions_B)
#     embeddings_final = model.encode(descriptions_final)
#     embeddings_model = model.encode(descriptions_model)

#     all_embeddings = np.vstack([embeddings_A, embeddings_B, embeddings_final, embeddings_model]).astype(np.float32)
#     all_text = descriptions_A + descriptions_B + descriptions_final + descriptions_model
#     all_ids = ids_A + ids_B + ids_final + ids_model
#     n_A = len(descriptions_A)
#     n_B = len(descriptions_B)
#     n_final = len(descriptions_final)
#     n_model = len(descriptions_model)
#     source = ['A'] * n_A + ['B'] * n_B + ['final'] * n_final + ['model'] * n_model

#     dot_matrix = all_embeddings @ all_embeddings.T
#     norms = np.sum(all_embeddings * all_embeddings, axis=1)
#     denom = norms[:, None] + norms[None, :] + 1e-8
#     sims = np.tanh(2.0 * dot_matrix / denom)
#     np.fill_diagonal(sims, 1.0)
#     dists = 1.0 - sims
#     dists = np.maximum(dists, 0.0)
#     np.fill_diagonal(dists, 0.0)

#     for n_neighbors in n_neighbors_values:
#         for min_dist in min_dist_values:
#             reducer = umap.UMAP(
#                 metric='precomputed',
#                 random_state=42,
#                 n_neighbors=n_neighbors,
#                 min_dist=min_dist,
#             )
#             embedding_2d = reducer.fit_transform(dists)
#             projection_cache[cache_key][(n_neighbors, min_dist)] = pd.DataFrame({
#                 'x': embedding_2d[:, 0],
#                 'y': embedding_2d[:, 1],
#                 'id': all_ids,
#                 'description': [two_line_hover(value, max_chars=max_chars) for value in all_text],
#                 'source': source,
#                 'avg_similarity': sims.mean(axis=1),
#             })


# rows_A_desc = df_stats_A[['id', 'intention_description']].dropna(subset=['id', 'intention_description'])
# rows_B_desc = df_stats_B[['id', 'intention_description']].dropna(subset=['id', 'intention_description'])
# rows_final_desc = df_stats_final[['id', 'intention_description']].dropna(subset=['id', 'intention_description'])
# rows_model_desc = df_model[['id', 'intention_description']].dropna(subset=['id', 'intention_description'])

# rows_A_exp = df_stats_A[['id', 'intention_explanation']].dropna(subset=['id', 'intention_explanation'])
# rows_B_exp = df_stats_B[['id', 'intention_explanation']].dropna(subset=['id', 'intention_explanation'])
# rows_final_exp = df_stats_final[['id', 'intention_explanation']].dropna(subset=['id', 'intention_explanation'])
# rows_model_exp = df_model[['id', 'intention_explanation']].dropna(subset=['id', 'intention_explanation'])

# print('Precomputing description projections...')
# compute_projection_cache(rows_A_desc, rows_B_desc, rows_final_desc, rows_model_desc, 'intention_description', 'description', max_chars=120)
# print('Precomputing explanation projections...')
# compute_projection_cache(rows_A_exp, rows_B_exp, rows_final_exp, rows_model_exp, 'intention_explanation', 'explanation', max_chars=120)
# print(f"Cached {len(projection_cache['description'])} description combinations and {len(projection_cache['explanation'])} explanation combinations.")


# def render_projection(plot_df, title):
#     fig = px.scatter(
#         plot_df,
#         x='x',
#         y='y',
#         color='source',
#         category_orders={'source': ['final', 'A', 'B', 'model']},
#         custom_data=['id', 'description', 'avg_similarity', 'source'],
#         color_discrete_map={'A': '#609AC1', 'B': '#C8C85A', 'final': '#ADE09E', 'model': '#D07EBD'},
#         title=title,
#         width=1000,
#         height=800,
#     )
#     fig.update_traces(
#         marker=dict(size=8, opacity=0.7),
#         hovertemplate='id=%{customdata[0]}<br>description=%{customdata[1]}<br>avg_similarity=%{customdata[2]:.3f}<br>source=%{customdata[3]}<extra></extra>'
#     )
#     fig.update_layout(
#         hoverlabel=dict(align='left'),
#         paper_bgcolor='white',
#         plot_bgcolor='white',
#         xaxis=dict(showgrid=False, zeroline=False),
#         yaxis=dict(showgrid=False, zeroline=False),
#     )
#     fig.show()


# neighbor_slider = widgets.SelectionSlider(
#     options=n_neighbors_values,
#     value=100,
#     description='n_neighbors',
#     continuous_update=False,
# )
# min_dist_slider = widgets.SelectionSlider(
#     options=min_dist_values,
#     value=0.25,
#     description='min_dist',
#     continuous_update=False,
# )
# plot_selector = widgets.ToggleButtons(
#     options=[('Descriptions', 'description'), ('Explanations', 'explanation')],
#     value='description',
#     description='Plot',
# )

# output = widgets.Output()


# def update_plot(*args):
#     with output:
#         clear_output(wait=True)
#         key = (neighbor_slider.value, min_dist_slider.value)
#         plot_df = projection_cache[plot_selector.value][key]
#         if plot_selector.value == 'description':
#             title = f"UMAP: Intention Descriptions (n_neighbors={neighbor_slider.value}, min_dist={min_dist_slider.value})"
#         else:
#             title = f"UMAP: Intention Explanations (n_neighbors={neighbor_slider.value}, min_dist={min_dist_slider.value})"
#         render_projection(plot_df, title)


# neighbor_slider.observe(update_plot, names='value')
# min_dist_slider.observe(update_plot, names='value')
# plot_selector.observe(update_plot, names='value')

# display(widgets.VBox([widgets.HBox([neighbor_slider, min_dist_slider]), plot_selector, output]))
# update_plot()

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import textwrap
import umap
import ipywidgets as widgets
from IPython.display import display, clear_output
from sentence_transformers import SentenceTransformer


def two_line_hover(text, max_chars=120):
    wrapped = textwrap.wrap(str(text), width=max_chars)
    if len(wrapped) <= 2:
        return '<br>'.join(wrapped)
    first = wrapped[0]
    second = ' '.join(wrapped[1:])
    return first + '<br>' + textwrap.fill(second, width=max_chars)

n_neighbors_values = [5, 10, 20, 50, 100, 200, 500]
min_dist_values = [0, 0.1, 0.25, 0.5, 0.75, 0.99]

bge_model = SentenceTransformer('BAAI/bge-large-en-v1.5')

projection_cache_bge = {'description': {}, 'explanation': {}}


def compute_projection_cache_bge(rows_A, rows_B, rows_final, rows_model, text_column, cache_key, max_chars=120):
    descriptions_A = rows_A[text_column].tolist()
    descriptions_B = rows_B[text_column].tolist()
    descriptions_final = rows_final[text_column].tolist()
    descriptions_model = rows_model[text_column].tolist()
    ids_A = rows_A['id'].tolist()
    ids_B = rows_B['id'].tolist()
    ids_final = rows_final['id'].tolist()
    ids_model = rows_model['id'].tolist()

    embeddings_A = bge_model.encode(descriptions_A)
    embeddings_B = bge_model.encode(descriptions_B)
    embeddings_final = bge_model.encode(descriptions_final)
    embeddings_model = bge_model.encode(descriptions_model)

    all_embeddings = np.vstack([embeddings_A, embeddings_B, embeddings_final, embeddings_model]).astype(np.float32)
    all_text = descriptions_A + descriptions_B + descriptions_final + descriptions_model
    all_ids = ids_A + ids_B + ids_final + ids_model
    n_A = len(descriptions_A)
    n_B = len(descriptions_B)
    n_final = len(descriptions_final)
    n_model = len(descriptions_model)
    source = ['A'] * n_A + ['B'] * n_B + ['final'] * n_final + ['model'] * n_model

    dot_matrix = all_embeddings @ all_embeddings.T
    norms = np.sum(all_embeddings * all_embeddings, axis=1)
    denom = norms[:, None] + norms[None, :] + 1e-8
    sims = np.tanh(2.0 * dot_matrix / denom)
    np.fill_diagonal(sims, 1.0)
    dists = 1.0 - sims
    dists = np.maximum(dists, 0.0)
    np.fill_diagonal(dists, 0.0)

    for n_neighbors in n_neighbors_values:
        for min_dist in min_dist_values:
            reducer = umap.UMAP(
                metric='precomputed',
                random_state=42,
                n_neighbors=n_neighbors,
                min_dist=min_dist,
            )
            embedding_2d = reducer.fit_transform(dists)
            projection_cache_bge[cache_key][(n_neighbors, min_dist)] = pd.DataFrame({
                'x': embedding_2d[:, 0],
                'y': embedding_2d[:, 1],
                'id': all_ids,
                'description': [two_line_hover(value, max_chars=max_chars) for value in all_text],
                'source': source,
                'avg_similarity': sims.mean(axis=1),
            })


rows_A_desc = df_stats_A[['id', 'intention_description']].dropna(subset=['id', 'intention_description'])
rows_B_desc = df_stats_B[['id', 'intention_description']].dropna(subset=['id', 'intention_description'])
rows_final_desc = df_stats_final[['id', 'intention_description']].dropna(subset=['id', 'intention_description'])
rows_model_desc = df_model[['id', 'intention_description']].dropna(subset=['id', 'intention_description'])

rows_A_exp = df_stats_A[['id', 'intention_explanation']].dropna(subset=['id', 'intention_explanation'])
rows_B_exp = df_stats_B[['id', 'intention_explanation']].dropna(subset=['id', 'intention_explanation'])
rows_final_exp = df_stats_final[['id', 'intention_explanation']].dropna(subset=['id', 'intention_explanation'])
rows_model_exp = df_model[['id', 'intention_explanation']].dropna(subset=['id', 'intention_explanation'])

print('Precomputing bge description projections...')
compute_projection_cache_bge(rows_A_desc, rows_B_desc, rows_final_desc, rows_model_desc, 'intention_description', 'description', max_chars=120)
print('Precomputing bge explanation projections...')
compute_projection_cache_bge(rows_A_exp, rows_B_exp, rows_final_exp, rows_model_exp, 'intention_explanation', 'explanation', max_chars=120)
print(f"Cached {len(projection_cache_bge['description'])} description combinations and {len(projection_cache_bge['explanation'])} explanation combinations.")


def render_projection_bge(plot_df, title):
    fig = px.scatter(
        plot_df,
        x='x',
        y='y',
        color='source',
        category_orders={'source': ['model','final', 'A', 'B']},
        custom_data=['id', 'description', 'avg_similarity', 'source'],
        color_discrete_map={'A': '#609AC1', 'B': '#C8C85A', 'final': '#ADE09E', 'model': '#D07EBD'},
        title=title,
        width=1000,
        height=800,
    )
    fig.update_traces(
        marker=dict(size=8, opacity=0.7),
        hovertemplate='id=%{customdata[0]}<br>description=%{customdata[1]}<br>avg_similarity=%{customdata[2]:.3f}<br>source=%{customdata[3]}<extra></extra>'
    )
    fig.update_layout(
        hoverlabel=dict(align='left'),
        paper_bgcolor='white',
        plot_bgcolor='white',
        xaxis=dict(showgrid=False, zeroline=False),
        yaxis=dict(showgrid=False, zeroline=False),
    )
    fig.show()


neighbor_slider_bge = widgets.SelectionSlider(
    options=n_neighbors_values,
    value=100,
    description='n_neighbors',
    continuous_update=False,
)
min_dist_slider_bge = widgets.SelectionSlider(
    options=min_dist_values,
    value=0.25,
    description='min_dist',
    continuous_update=False,
)
plot_selector_bge = widgets.ToggleButtons(
    options=[('Descriptions', 'description'), ('Explanations', 'explanation')],
    value='description',
    description='Plot',
)

output_bge = widgets.Output()


def update_plot_bge(*args):
    with output_bge:
        clear_output(wait=True)
        key = (neighbor_slider_bge.value, min_dist_slider_bge.value)
        plot_df = projection_cache_bge[plot_selector_bge.value][key]
        if plot_selector_bge.value == 'description':
            title = f"BGE UMAP: Intention Descriptions (n_neighbors={neighbor_slider_bge.value}, min_dist={min_dist_slider_bge.value})"
        else:
            title = f"BGE UMAP: Intention Explanations (n_neighbors={neighbor_slider_bge.value}, min_dist={min_dist_slider_bge.value})"
        render_projection_bge(plot_df, title)


neighbor_slider_bge.observe(update_plot_bge, names='value')
min_dist_slider_bge.observe(update_plot_bge, names='value')
plot_selector_bge.observe(update_plot_bge, names='value')

display(widgets.VBox([widgets.HBox([neighbor_slider_bge, min_dist_slider_bge]), plot_selector_bge, output_bge]))
update_plot_bge()

In [ ]:
# import numpy as np
# import pandas as pd
# import plotly.express as px
# import textwrap
# import umap
# import ipywidgets as widgets
# from IPython.display import display, clear_output


# def two_line_hover(text, max_chars=120):
#     wrapped = textwrap.wrap(str(text), width=max_chars)
#     if len(wrapped) <= 2:
#         return '<br>'.join(wrapped)
#     first = wrapped[0]
#     second = ' '.join(wrapped[1:])
#     return first + '<br>' + textwrap.fill(second, width=max_chars)


# def build_combined_text(description, explanation):
#     return f"Description: {description}\n\nExplanation: {explanation}"

# n_neighbors_values = [5, 10, 20, 50, 100, 200, 500]
# min_dist_values = [0, 0.1, 0.25, 0.5, 0.75, 0.99]

# projection_cache_bge_merged = {}


# def compute_projection_cache_bge_merged(rows_A, rows_B, rows_final, rows_model, cache_key, max_chars=120):
#     rows_A = rows_A.dropna(subset=['id', 'intention_description', 'intention_explanation'])
#     rows_B = rows_B.dropna(subset=['id', 'intention_description', 'intention_explanation'])
#     rows_final = rows_final.dropna(subset=['id', 'intention_description', 'intention_explanation'])
#     rows_model = rows_model.dropna(subset=['id', 'intention_description', 'intention_explanation'])

#     combined_A = [build_combined_text(desc, exp) for desc, exp in zip(rows_A['intention_description'], rows_A['intention_explanation'])]
#     combined_B = [build_combined_text(desc, exp) for desc, exp in zip(rows_B['intention_description'], rows_B['intention_explanation'])]
#     combined_final = [build_combined_text(desc, exp) for desc, exp in zip(rows_final['intention_description'], rows_final['intention_explanation'])]
#     combined_model = [build_combined_text(desc, exp) for desc, exp in zip(rows_model['intention_description'], rows_model['intention_explanation'])]

#     ids_A = rows_A['id'].tolist()
#     ids_B = rows_B['id'].tolist()
#     ids_final = rows_final['id'].tolist()
#     ids_model = rows_model['id'].tolist()

#     embeddings_A = bge_model.encode(combined_A)
#     embeddings_B = bge_model.encode(combined_B)
#     embeddings_final = bge_model.encode(combined_final)
#     embeddings_model = bge_model.encode(combined_model)

#     all_embeddings = np.vstack([embeddings_A, embeddings_B, embeddings_final, embeddings_model]).astype(np.float32)
#     all_text = combined_A + combined_B + combined_final + combined_model
#     all_ids = ids_A + ids_B + ids_final + ids_model
#     n_A = len(combined_A)
#     n_B = len(combined_B)
#     n_final = len(combined_final)
#     n_model = len(combined_model)
#     source = ['A'] * n_A + ['B'] * n_B + ['final'] * n_final + ['model'] * n_model

#     dot_matrix = all_embeddings @ all_embeddings.T
#     norms = np.sum(all_embeddings * all_embeddings, axis=1)
#     denom = norms[:, None] + norms[None, :] + 1e-8
#     sims = np.tanh(2.0 * dot_matrix / denom)
#     np.fill_diagonal(sims, 1.0)
#     dists = 1.0 - sims
#     dists = np.maximum(dists, 0.0)
#     np.fill_diagonal(dists, 0.0)

#     for n_neighbors in n_neighbors_values:
#         for min_dist in min_dist_values:
#             reducer = umap.UMAP(
#                 metric='precomputed',
#                 random_state=42,
#                 n_neighbors=n_neighbors,
#                 min_dist=min_dist,
#             )
#             embedding_2d = reducer.fit_transform(dists)
#             projection_cache_bge_merged[(cache_key, n_neighbors, min_dist)] = pd.DataFrame({
#                 'x': embedding_2d[:, 0],
#                 'y': embedding_2d[:, 1],
#                 'id': all_ids,
#                 'description': [two_line_hover(value, max_chars=max_chars) for value in all_text],
#                 'source': source,
#                 'avg_similarity': sims.mean(axis=1),
#             })


# rows_A_merged = df_stats_A[['id', 'intention_description', 'intention_explanation']]
# rows_B_merged = df_stats_B[['id', 'intention_description', 'intention_explanation']]
# rows_final_merged = df_stats_final[['id', 'intention_description', 'intention_explanation']]
# rows_model_merged = df_model[['id', 'intention_description', 'intention_explanation']]

# print('Precomputing bge merged projections...')
# compute_projection_cache_bge_merged(rows_A_merged, rows_B_merged, rows_final_merged, rows_model_merged, 'merged', max_chars=120)
# print(f"Cached {len(n_neighbors_values) * len(min_dist_values)} merged combinations.")


# def render_projection_bge_merged(plot_df, title):
#     fig = px.scatter(
#         plot_df,
#         x='x',
#         y='y',
#         color='source',
#         category_orders={'source': ['model', 'final', 'A', 'B']},
#         custom_data=['id', 'description', 'avg_similarity', 'source'],
#         color_discrete_map={'A': '#609AC1', 'B': '#C8C85A', 'final': '#ADE09E', 'model': '#D07EBD'},
#         title=title,
#         width=1000,
#         height=800,
#     )
#     fig.update_traces(
#         marker=dict(size=8, opacity=0.7),
#         hovertemplate='id=%{customdata[0]}<br>description=%{customdata[1]}<br>avg_similarity=%{customdata[2]:.3f}<br>source=%{customdata[3]}<extra></extra>'
#     )
#     fig.update_layout(
#         hoverlabel=dict(align='left'),
#         paper_bgcolor='white',
#         plot_bgcolor='white',
#         xaxis=dict(showgrid=False, zeroline=False),
#         yaxis=dict(showgrid=False, zeroline=False),
#     )
#     fig.show()


# neighbor_slider_bge_merged = widgets.SelectionSlider(
#     options=n_neighbors_values,
#     value=100,
#     description='n_neighbors',
#     continuous_update=False,
# )
# min_dist_slider_bge_merged = widgets.SelectionSlider(
#     options=min_dist_values,
#     value=0.25,
#     description='min_dist',
#     continuous_update=False,
# )
# output_bge_merged = widgets.Output()


# def update_plot_bge_merged(*args):
#     with output_bge_merged:
#         clear_output(wait=True)
#         key = ('merged', neighbor_slider_bge_merged.value, min_dist_slider_bge_merged.value)
#         plot_df = projection_cache_bge_merged[key]
#         title = f"BGE UMAP: Intention Description + Explanation (n_neighbors={neighbor_slider_bge_merged.value}, min_dist={min_dist_slider_bge_merged.value})"
#         render_projection_bge_merged(plot_df, title)


# neighbor_slider_bge_merged.observe(update_plot_bge_merged, names='value')
# min_dist_slider_bge_merged.observe(update_plot_bge_merged, names='value')

# display(widgets.VBox([widgets.HBox([neighbor_slider_bge_merged, min_dist_slider_bge_merged]), output_bge_merged]))
# update_plot_bge_merged()